# 🏟️ LaLiga Team Playing Style — Feature Engineering & Style Scoring

---

## 📋 Project Overview

**Tujuan Notebook:**  
Notebook ini merupakan tahap persiapan data untuk analisis clustering gaya bermain tim LaLiga.  
Tujuan utamanya adalah:

1. **Feature Selection** — Memilih fitur-fitur yang merepresentasikan gaya bermain tim secara konseptual (bukan random).
2. **Feature Engineering** — Menormalisasi fitur terpilih menggunakan Min-Max Scaling (skala 0–100).
3. **Style Scoring** — Menghitung skor komposit untuk 9 dimensi gaya bermain.

**Input:**
- Tabel `gold.teams_statistics` dari PostgreSQL — One Big Table (OBT) berisi 124+ kolom statistik 20 tim LaLiga musim 2024/25.

**Output:**
- DataFrame `teams_ml_features` berisi **club + 9 style scores** (skala 0–100) yang siap digunakan pada notebook clustering berikutnya.

**Catatan Penting:**
> ⚠️ Notebook ini **TIDAK** melakukan clustering. Notebook ini hanya menghasilkan dataset fitur yang siap digunakan oleh notebook clustering berikutnya.

---

### 9 Dimensi Gaya Bermain

| # | Style Score | Tujuan |
|---|------------|--------|
| 1 | **Possession Score** | Mengukur dominasi penguasaan bola |
| 2 | **Direct Play Score** | Mengukur kecenderungan bermain langsung |
| 3 | **Pressing Score** | Mengukur intensitas pressing |
| 4 | **Attacking Efficiency Score** | Mengukur efektivitas penyelesaian serangan |
| 5 | **Defensive Solidity Score** | Mengukur kekuatan bertahan |
| 6 | **Set Piece Score** | Mengukur ancaman dari bola mati |
| 7 | **Chance Creation Score** | Mengukur kemampuan menciptakan peluang |
| 8 | **Transition Score** | Mengukur kecepatan & efektivitas transisi |
| 9 | **Discipline Score** | Mengukur kedisiplinan (inverted — semakin sedikit pelanggaran, semakin tinggi skor) |

---

## 1. Import Library

Library yang digunakan:
- **pandas** & **numpy** — manipulasi data dan komputasi numerik
- **matplotlib** — visualisasi statis (radar chart)
- **plotly** — visualisasi interaktif
- **sqlalchemy** — koneksi ke PostgreSQL
- **sklearn.preprocessing** — MinMaxScaler untuk normalisasi
- **dotenv** — membaca credential database dari file `.env`

In [ ]:
# === Standard Library ===
import os
import warnings
from pathlib import Path

# === Data Manipulation ===
import numpy as np
import pandas as pd

# === Visualization ===
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# === Database ===
from sqlalchemy import create_engine
from dotenv import load_dotenv

# === Preprocessing ===
from sklearn.preprocessing import MinMaxScaler

# === Settings ===
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Semua library berhasil diimpor.")

---

## 2. Load Data dari PostgreSQL

Data dimuat langsung dari tabel **`gold.teams_statistics`** di PostgreSQL.  
Tabel ini merupakan *One Big Table (OBT)* yang sudah menggabungkan 12 sub-tabel statistik tim LaLiga menjadi satu tabel dengan 124+ kolom.

Koneksi database menggunakan **SQLAlchemy** dengan credential yang dibaca dari file `.env`.

In [ ]:
# --- Konfigurasi Database ---
# Membaca credential dari .env di root project
PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")

DB_URL = (
    f"postgresql://{os.getenv('DB_USER', 'g')}:{os.getenv('DB_PASSWORD', '')}"
    f"@{os.getenv('DB_HOST', 'localhost')}:{os.getenv('DB_PORT', '5432')}"
    f"/{os.getenv('DB_NAME', 'laliga')}"
)

engine = create_engine(DB_URL)
print(f"✅ Engine created: {engine.url.database}@{engine.url.host}")

In [ ]:
# --- Load Gold OBT ---
query = "SELECT * FROM gold.teams_statistics ORDER BY club"
df_raw = pd.read_sql(query, engine)

print(f"Shape: {df_raw.shape}  ({df_raw.shape[0]} tim, {df_raw.shape[1]} kolom)")
print(f"Clubs: {df_raw['club'].nunique()} tim unik")

In [ ]:
# --- Preview Data (5 baris pertama) ---
df_raw.head()

In [ ]:
# --- Informasi Tipe Data & Memory ---
df_raw.info()

In [ ]:
# --- Statistik Deskriptif ---
df_raw.describe().T

---

## 3. Feature Understanding

Tabel `gold.teams_statistics` memiliki **124+ kolom**. Namun, **tidak semua kolom relevan** untuk mengukur gaya bermain tim.

### Prinsip Pemilihan Feature

Feature dipilih **secara konseptual** berdasarkan:
1. **Representasi gaya bermain** — Setiap feature harus mencerminkan aspek spesifik dari cara sebuah tim bermain.
2. **Interpretabilitas** — Feature harus bisa dijelaskan secara logis dalam konteks sepakbola.
3. **Menghindari redundansi** — Jika dua kolom mengukur hal yang sama, cukup pilih salah satu.
4. **Per-game metrics lebih diutamakan** — Untuk metrik absolut (jumlah total), diutamakan versi *per game* agar perbandingan antar-tim lebih fair.

### Kolom yang TIDAK dipakai
- `played` — Semua tim bermain 38 pertandingan (konstan, tidak informatif).
- Kolom redundan — Misalnya `att_goals` dan `att_goals_per_game` mengukur hal yang sama; cukup pilih salah satu.
- Kolom identitas — `club` dipakai sebagai label, bukan fitur.

In [ ]:
# --- Daftar Semua Kolom ---
print(f"Total kolom: {len(df_raw.columns)}\n")
for i, col in enumerate(df_raw.columns, 1):
    print(f"  {i:3d}. {col}")

---

## 4. Feature Selection

Feature dikelompokkan berdasarkan **9 dimensi gaya bermain**.  
Setiap kelompok merepresentasikan satu aspek taktis dari cara sebuah tim bermain sepakbola.

### A. Possession Play 🎯
Mengukur **dominasi penguasaan bola** — seberapa besar porsi penguasaan bola, akurasi passing, dan kesabaran membangun serangan.

| Feature (kolom di DB) | Deskripsi |
|---|---|
| `def_avg_possession_pct` | Rata-rata penguasaan bola (%) |
| `pas_passes_total` | Total passing per musim |
| `pas_passes_successful` | Passing berhasil per musim |
| `pas_passes_pct` | Akurasi passing (%) |
| `pas_passes_per_game` | Passing per pertandingan |
| `pas_final_third_successful` | Passing berhasil di sepertiga akhir |
| `seq_sequence_time` | Rata-rata durasi sekuens serangan (detik) |
| `seq_buildups_total` | Total build-up play |
| `seq_passes_per_seq` | Rata-rata passing per sekuens serangan |

### B. Direct Play ⚡
Mengukur **kecenderungan bermain langsung** — serangan cepat, umpan terobosan, dan efisiensi per tembakan.

| Feature | Deskripsi |
|---|---|
| `seq_direct_speed` | Kecepatan serangan langsung |
| `seq_direct_attacks_total` | Total serangan langsung |
| `seq_direct_attacks_goals` | Gol dari serangan langsung |
| `pas_through_balls` | Total umpan terobosan |
| `pas_crosses_total` | Total umpan silang |
| `att_xg_per_shot` | Expected Goals per tembakan |

### C. High Press 🏃
Mengukur **intensitas pressing** — seberapa tinggi dan agresif tim menekan lawan saat tidak menguasai bola.

| Feature | Deskripsi |
|---|---|
| `prs_pressed_seqs` | Total sekuens pressing |
| `prs_high_turnovers_total` | Total *high turnover* (rebut bola di area tinggi) |
| `prs_high_turnovers_shot_pct` | % *high turnover* yang berujung tembakan |
| `prs_start_distance` | Jarak rata-rata mulai pressing dari gawang lawan |
| `prs_ppda` | Passes Per Defensive Action (semakin rendah = pressing lebih intens) |

### D. Attacking Efficiency ⚽
Mengukur **efektivitas penyelesaian serangan** — konversi peluang menjadi gol.

| Feature | Deskripsi |
|---|---|
| `att_goals` | Total gol |
| `att_xg` | Expected Goals |
| `att_goals_vs_xg` | Selisih gol vs xG (overperformance) |
| `att_conversion_pct` | Persentase konversi tembakan → gol |
| `att_shots` | Total tembakan |
| `att_sot` | Total tembakan tepat sasaran |
| `att_touches_in_box` | Total sentuhan di dalam kotak penalti |

### E. Defensive Solidity 🛡️
Mengukur **kekuatan bertahan** — kemampuan mencegah lawan mencetak gol.

| Feature | Deskripsi |
|---|---|
| `def_tackles` | Total tekel |
| `def_interceptions` | Total intersepsi |
| `def_possession_won` | Total rebut bola |
| `def_blocks` | Total blok tembakan |
| `def_clearances` | Total sapuan bola |
| `def_goals_conceded` | Total gol kebobolan (inverted) |
| `def_xg_against` | xG lawan (inverted) |
| `def_goals_vs_xg_against` | Selisih gol kebobolan vs xG lawan (inverted) |

### F. Set Piece 🎯
Mengukur **ancaman dari bola mati** — baik menyerang maupun bertahan.

| Feature | Deskripsi |
|---|---|
| `att_sp_goals` | Gol dari set-piece (menyerang) |
| `att_sp_xg` | xG dari set-piece (menyerang) |
| `def_sp_goals` | Gol kebobolan dari set-piece (inverted) |
| `def_sp_xg` | xG kebobolan dari set-piece (inverted) |

### G. Chance Creation 🎨
Mengukur **kemampuan menciptakan peluang** — kombinasi sentuhan, umpan, dan tembakan.

| Feature | Deskripsi |
|---|---|
| `att_touches_in_box` | Sentuhan di kotak penalti |
| `pas_through_balls` | Umpan terobosan |
| `pas_final_third_successful` | Passing berhasil di sepertiga akhir |
| `att_shots` | Total tembakan |
| `att_xg` | Expected Goals |

### H. Transition ⏩
Mengukur **kecepatan dan efektivitas transisi** dari bertahan ke menyerang.

| Feature | Deskripsi |
|---|---|
| `seq_direct_speed` | Kecepatan serangan langsung |
| `seq_direct_attacks_total` | Total serangan langsung |
| `att_fast_breaks_total` | Total serangan balik cepat |
| `prs_high_turnovers_total` | High turnover (rebut bola tinggi) |
| `seq_sequence_time` | Durasi sekuens (inverted — semakin cepat, semakin baik transisi) |

### I. Discipline 📏
Mengukur **kedisiplinan** — semakin sedikit pelanggaran dan kartu, semakin tinggi skor.

| Feature | Deskripsi |
|---|---|
| `msc_fouls` | Total pelanggaran (inverted) |
| `msc_yellows` | Total kartu kuning (inverted) |
| `msc_reds` | Total kartu merah (inverted) |
| `msc_pens_conceded` | Total penalti diberikan (inverted) |

> **Catatan:** Feature bertanda *(inverted)* artinya nilai tinggi pada kolom asli menunjukkan performa **buruk**, sehingga skor perlu dibalik agar logika skoring konsisten (skor tinggi = bagus).

In [ ]:
# ===========================================================================
# Definisi Feature Groups
# ===========================================================================
# Setiap key = nama dimensi gaya bermain
# Setiap value = dict berisi:
#   - 'features' : list kolom yang digunakan
#   - 'inverted' : list kolom yang perlu dibalik (tinggi = buruk → rendah = bagus)

STYLE_FEATURES = {
    "Possession": {
        "features": [
            "def_avg_possession_pct",
            "pas_passes_total",
            "pas_passes_successful",
            "pas_passes_pct",
            "pas_passes_per_game",
            "pas_final_third_successful",
            "seq_sequence_time",
            "seq_buildups_total",
            "seq_passes_per_seq",
        ],
        "inverted": [],
    },
    "Direct Play": {
        "features": [
            "seq_direct_speed",
            "seq_direct_attacks_total",
            "seq_direct_attacks_goals",
            "pas_through_balls",
            "pas_crosses_total",
            "att_xg_per_shot",
        ],
        "inverted": [],
    },
    "Pressing": {
        "features": [
            "prs_pressed_seqs",
            "prs_high_turnovers_total",
            "prs_high_turnovers_shot_pct",
            "prs_start_distance",
            "prs_ppda",
        ],
        "inverted": ["prs_ppda"],  # PPDA rendah = pressing lebih intens
    },
    "Attacking Efficiency": {
        "features": [
            "att_goals",
            "att_xg",
            "att_goals_vs_xg",
            "att_conversion_pct",
            "att_shots",
            "att_sot",
            "att_touches_in_box",
        ],
        "inverted": [],
    },
    "Defensive Solidity": {
        "features": [
            "def_tackles",
            "def_interceptions",
            "def_possession_won",
            "def_blocks",
            "def_clearances",
            "def_goals_conceded",
            "def_xg_against",
            "def_goals_vs_xg_against",
        ],
        "inverted": ["def_goals_conceded", "def_xg_against", "def_goals_vs_xg_against"],
    },
    "Set Piece": {
        "features": [
            "att_sp_goals",
            "att_sp_xg",
            "def_sp_goals",
            "def_sp_xg",
        ],
        "inverted": ["def_sp_goals", "def_sp_xg"],  # Kebobolan dari set-piece = buruk
    },
    "Chance Creation": {
        "features": [
            "att_touches_in_box",
            "pas_through_balls",
            "pas_final_third_successful",
            "att_shots",
            "att_xg",
        ],
        "inverted": [],
    },
    "Transition": {
        "features": [
            "seq_direct_speed",
            "seq_direct_attacks_total",
            "att_fast_breaks_total",
            "prs_high_turnovers_total",
            "seq_sequence_time",
        ],
        "inverted": ["seq_sequence_time"],  # Durasi rendah = transisi cepat
    },
    "Discipline": {
        "features": [
            "msc_fouls",
            "msc_yellows",
            "msc_reds",
            "msc_pens_conceded",
        ],
        "inverted": ["msc_fouls", "msc_yellows", "msc_reds", "msc_pens_conceded"],
    },
}

# --- Ringkasan Feature Selection ---
print("📊 Ringkasan Feature Selection:")
print("=" * 50)
total_features = 0
for style, config in STYLE_FEATURES.items():
    n = len(config["features"])
    inv = len(config["inverted"])
    total_features += n
    inv_str = f"  ({inv} inverted)" if inv > 0 else ""
    print(f"  {style:25s}  →  {n} features{inv_str}")
print("=" * 50)
print(f"  Total features terpilih: {total_features}")

---

## 5. Validasi Feature

Sebelum melanjutkan, kita perlu memastikan bahwa semua kolom yang dipilih **benar-benar ada** di dalam dataset.  
Ini adalah langkah penting untuk menghindari error di tahap selanjutnya.

In [ ]:
# --- Validasi: semua feature harus ada di df_raw ---
all_features = set()
for style, config in STYLE_FEATURES.items():
    all_features.update(config["features"])

missing = all_features - set(df_raw.columns)

if missing:
    print(f"❌ Feature TIDAK ditemukan di dataset: {missing}")
else:
    print(f"✅ Semua {len(all_features)} feature unik ditemukan di dataset.")

In [ ]:
# --- Preview: nilai feature terpilih untuk 5 tim pertama ---
preview_cols = ["club"] + sorted(all_features)
df_raw[preview_cols].head()

---

## 6. Feature Engineering — Min-Max Normalization

### Mengapa Normalisasi Diperlukan?

Feature yang kita pilih memiliki **skala yang berbeda-beda**:
- `def_avg_possession_pct` → skala 40–70 (%)
- `pas_passes_total` → skala 10.000–25.000 (jumlah absolut)
- `att_conversion_pct` → skala 5–20 (%)

Jika langsung dirata-ratakan tanpa normalisasi, feature dengan skala besar akan **mendominasi** skor.

### Metode: Min-Max Scaling ke skala 0–100

$$X_{scaled} = \frac{X - X_{min}}{X_{max} - X_{min}} \times 100$$

Dimana:
- $X_{min}$ = nilai minimum feature di dataset (20 tim)
- $X_{max}$ = nilai maksimum feature di dataset (20 tim)
- Hasil: skor **0** (terburuk) sampai **100** (terbaik) untuk setiap feature

### Penanganan Feature Inverted

Beberapa feature memiliki logika terbalik (nilai tinggi = buruk), contoh:
- `def_goals_conceded`: kebobolan banyak = buruk
- `msc_fouls`: banyak pelanggaran = buruk
- `prs_ppda`: PPDA tinggi = pressing kurang intens

Untuk feature ini, kita **membalik** nilainya setelah normalisasi:

$$X_{inverted} = 100 - X_{scaled}$$

In [ ]:
# ===========================================================================
# Fungsi Normalisasi Min-Max (0-100)
# ===========================================================================

def normalize_features(
    df: pd.DataFrame,
    features: list[str],
    inverted: list[str] | None = None,
) -> pd.DataFrame:
    """Normalisasi kolom-kolom terpilih ke skala 0-100 menggunakan Min-Max Scaling.

    Args:
        df: DataFrame sumber (tidak diubah).
        features: List nama kolom yang akan dinormalisasi.
        inverted: List nama kolom yang perlu dibalik (tinggi = buruk).

    Returns:
        DataFrame baru berisi kolom ternormalisasi (0-100).
    """
    if inverted is None:
        inverted = []

    scaler = MinMaxScaler(feature_range=(0, 100))
    df_norm = pd.DataFrame(
        scaler.fit_transform(df[features]),
        columns=features,
        index=df.index,
    )

    # Balik kolom inverted: 100 - nilai
    for col in inverted:
        if col in df_norm.columns:
            df_norm[col] = 100 - df_norm[col]

    return df_norm


print("✅ Fungsi normalize_features() siap digunakan.")

In [ ]:
# ===========================================================================
# Normalisasi seluruh feature terpilih
# ===========================================================================

# Kumpulkan semua feature unik beserta inverted-nya
all_features_list = sorted(all_features)
all_inverted = set()
for config in STYLE_FEATURES.values():
    all_inverted.update(config["inverted"])

# Normalisasi
df_normalized = normalize_features(
    df_raw,
    features=all_features_list,
    inverted=list(all_inverted),
)

# Tambahkan kolom club
df_normalized.insert(0, "club", df_raw["club"].values)

print(f"✅ Normalisasi selesai: {df_normalized.shape}")
print(f"   Range: {df_normalized.drop(columns='club').min().min():.0f} – {df_normalized.drop(columns='club').max().max():.0f}")
df_normalized.head()

---

## 7. Style Scoring

### Pendekatan: Simple Average

Skor untuk setiap dimensi gaya bermain dihitung sebagai **rata-rata (mean)** dari seluruh feature yang sudah dinormalisasi dalam kelompok tersebut.

$$\text{Style Score} = \frac{1}{n} \sum_{i=1}^{n} X_{i,scaled}$$

**Mengapa simple average?**
1. **Mudah diinterpretasi** — Setiap feature berkontribusi sama rata.
2. **Reproducible** — Tidak ada bobot subjektif yang bisa diperdebatkan.
3. **Sesuai untuk skripsi** — Penjelasan sederhana, tidak perlu justifikasi bobot kompleks.

> Jika diperlukan, bobot bisa ditambahkan di iterasi berikutnya berdasarkan domain expertise atau analisis korelasi.

In [ ]:
# ===========================================================================
# Hitung Style Scores
# ===========================================================================

def compute_style_scores(
    df_norm: pd.DataFrame,
    style_config: dict,
) -> pd.DataFrame:
    """Hitung skor komposit per dimensi gaya bermain.

    Setiap skor merupakan rata-rata dari feature ternormalisasi
    dalam kelompok tersebut.

    Args:
        df_norm: DataFrame berisi kolom ternormalisasi (0-100) + kolom 'club'.
        style_config: Dict konfigurasi STYLE_FEATURES.

    Returns:
        DataFrame baru berisi 'club' + skor per dimensi (0-100).
    """
    scores = {"club": df_norm["club"].values}

    for style_name, config in style_config.items():
        col_name = f"{style_name} Score"
        features = config["features"]
        scores[col_name] = df_norm[features].mean(axis=1).round(2)

    return pd.DataFrame(scores)


# --- Hitung semua skor ---
df_scores = compute_style_scores(df_normalized, STYLE_FEATURES)

print("✅ Style Scores berhasil dihitung!")
print(f"   Shape: {df_scores.shape}")
print(f"   Kolom: {list(df_scores.columns)}")
print()
df_scores.sort_values("Possession Score", ascending=False).reset_index(drop=True)

---

### 📊 Statistik Deskriptif Style Scores

Mari lihat distribusi skor untuk setiap dimensi gaya bermain.

In [ ]:
# --- Statistik Deskriptif ---
score_cols = [c for c in df_scores.columns if c != "club"]
df_scores[score_cols].describe().round(2)

---

## 8. Interpretasi — Rankings

Setelah skor dihitung, kita bisa melihat **ranking** tim untuk setiap dimensi gaya bermain.  
Ini membantu memvalidasi bahwa skor yang dihasilkan **masuk akal secara sepakbola**.

Contoh validasi intuitif:
- Barcelona dan Real Madrid diharapkan **tinggi** di Possession Score.
- Tim yang bermain agresif diharapkan **tinggi** di Pressing Score.
- Tim papan bawah mungkin **lebih rendah** di Attacking Efficiency.

In [ ]:
# ===========================================================================
# Fungsi untuk menampilkan Top-N Ranking
# ===========================================================================

def show_rankings(
    df: pd.DataFrame,
    score_columns: list[str],
    top_n: int = 10,
) -> None:
    """Tampilkan ranking Top-N untuk setiap score column."""
    for col in score_columns:
        ranked = (
            df[["club", col]]
            .sort_values(col, ascending=False)
            .reset_index(drop=True)
        )
        ranked.index = ranked.index + 1  # Start from 1
        ranked.index.name = "Rank"
        print(f"\n🏆 Top {top_n} — {col}")
        print("─" * 40)
        print(ranked.head(top_n).to_string())
        print()

In [ ]:
# --- Tampilkan Top 10 untuk setiap dimensi ---
show_rankings(df_scores, score_cols, top_n=10)

---

### 📊 Visualisasi Heatmap — Semua Skor

Heatmap ini memberikan gambaran **komparatif** seluruh 20 tim di 9 dimensi gaya bermain sekaligus.  
Warna lebih gelap menunjukkan skor lebih tinggi (lebih dominan di dimensi tersebut).

In [ ]:
# --- Heatmap: Semua Style Scores ---
df_heatmap = df_scores.set_index("club")[score_cols].sort_values(
    "Possession Score", ascending=True
)

fig = px.imshow(
    df_heatmap,
    text_auto=".1f",
    color_continuous_scale="RdYlGn",
    aspect="auto",
    title="🗺️ Heatmap — Style Scores Seluruh Tim LaLiga",
    labels={"x": "Style Dimension", "y": "Club", "color": "Score"},
)

fig.update_layout(
    width=1000,
    height=700,
    font=dict(size=11),
    title_font_size=16,
    xaxis_tickangle=-30,
)

fig.show()

---

## 9. Radar Chart — Profil Gaya Bermain per Klub

Radar chart (spider chart) menampilkan **profil multidimensi** gaya bermain sebuah tim.  
Setiap sumbu merepresentasikan satu dimensi style score.

Tim dengan area radar yang luas dan merata menunjukkan **keseimbangan di banyak aspek**.  
Tim dengan tonjolan di satu sisi menunjukkan **spesialisasi** pada gaya bermain tertentu.

In [ ]:
# ===========================================================================
# Fungsi Radar Chart menggunakan Plotly
# ===========================================================================

def radar_chart(
    df: pd.DataFrame,
    club: str,
    score_columns: list[str],
    color: str = "#636EFA",
) -> go.Figure:
    """Buat radar chart untuk satu klub.

    Args:
        df: DataFrame berisi kolom 'club' dan kolom-kolom skor.
        club: Nama klub yang ingin ditampilkan.
        score_columns: List kolom skor untuk sumbu radar.
        color: Warna fill area radar.

    Returns:
        Plotly Figure object.
    """
    row = df[df["club"] == club].iloc[0]
    values = [row[col] for col in score_columns]
    # Tutup radar (titik pertama diulang di akhir)
    values += [values[0]]
    labels = [col.replace(" Score", "") for col in score_columns]
    labels += [labels[0]]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=labels,
        fill="toself",
        fillcolor=color,
        opacity=0.3,
        line=dict(color=color, width=2),
        name=club,
    ))

    fig.update_layout(
        polar=dict(
            radialaxis=dict(visible=True, range=[0, 100], ticksuffix=""),
        ),
        title=dict(
            text=f"🕸️ Radar Chart — {club}",
            font=dict(size=16),
        ),
        showlegend=False,
        width=600,
        height=500,
    )

    return fig


def radar_chart_comparison(
    df: pd.DataFrame,
    clubs: list[str],
    score_columns: list[str],
    colors: list[str] | None = None,
) -> go.Figure:
    """Buat radar chart perbandingan untuk beberapa klub."""
    if colors is None:
        colors = px.colors.qualitative.Plotly[:len(clubs)]

    labels = [col.replace(" Score", "") for col in score_columns]
    labels_closed = labels + [labels[0]]

    fig = go.Figure()
    for club, color in zip(clubs, colors):
        row = df[df["club"] == club].iloc[0]
        values = [row[col] for col in score_columns] + [row[score_columns[0]]]
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=labels_closed,
            fill="toself",
            opacity=0.2,
            line=dict(color=color, width=2),
            name=club,
        ))

    fig.update_layout(
        polar=dict(
            radialaxis=dict(visible=True, range=[0, 100]),
        ),
        title=dict(
            text=f"🕸️ Comparison — {' vs '.join(clubs)}",
            font=dict(size=16),
        ),
        width=700,
        height=550,
    )

    return fig


print("✅ Fungsi radar_chart() dan radar_chart_comparison() siap digunakan.")

### 🔍 Radar Chart — Contoh Klub Individu

In [ ]:
# --- Radar Chart: Barcelona ---
radar_chart(df_scores, "Barcelona", score_cols, color="#A50044").show()

In [ ]:
# --- Radar Chart: Real Madrid ---
radar_chart(df_scores, "Real Madrid", score_cols, color="#FEBE10").show()

In [ ]:
# --- Radar Chart: Atlético Madrid ---
radar_chart(df_scores, "Atlético Madrid", score_cols, color="#CE3524").show()

### 🔄 Radar Chart — Perbandingan Big Three LaLiga

In [ ]:
# --- Comparison: Barcelona vs Real Madrid vs Atlético Madrid ---
radar_chart_comparison(
    df_scores,
    clubs=["Barcelona", "Real Madrid", "Atlético Madrid"],
    score_columns=score_cols,
    colors=["#A50044", "#FEBE10", "#CE3524"],
).show()

### 🕸️ Radar Chart — Seluruh 20 Klub (Grid)

Tampilan grid radar chart untuk seluruh 20 tim LaLiga, memudahkan perbandingan visual secara menyeluruh.

In [ ]:
# --- Grid Radar Chart: Seluruh 20 Klub (Matplotlib) ---
clubs_sorted = df_scores.sort_values("Possession Score", ascending=False)["club"].tolist()
labels_short = [c.replace(" Score", "") for c in score_cols]
num_vars = len(score_cols)

# Sudut untuk setiap sumbu radar
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # Tutup lingkaran

fig, axes = plt.subplots(4, 5, figsize=(22, 18), subplot_kw=dict(polar=True))
fig.suptitle("🕸️ Radar Chart — Profil Gaya Bermain Seluruh Klub LaLiga", fontsize=18, y=1.02)

for idx, (ax, club) in enumerate(zip(axes.flat, clubs_sorted)):
    row = df_scores[df_scores["club"] == club].iloc[0]
    values = [row[col] for col in score_cols]
    values += values[:1]  # Tutup lingkaran

    ax.fill(angles, values, alpha=0.25, color="#636EFA")
    ax.plot(angles, values, color="#636EFA", linewidth=1.5)
    ax.set_ylim(0, 100)
    ax.set_yticks([25, 50, 75])
    ax.set_yticklabels(["25", "50", "75"], fontsize=6, color="gray")
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels_short, fontsize=7)
    ax.set_title(club, fontsize=10, fontweight="bold", pad=15)

plt.tight_layout()
plt.show()

---

## 10. Output Dataset — `teams_ml_features`

Tahap terakhir: menyusun dataset akhir yang siap digunakan oleh **notebook clustering berikutnya**.

Dataset ini berisi:
- `club` — Nama klub (label/identifier, bukan fitur)
- 9 kolom **Style Score** (skala 0–100)

Dataset ini akan menjadi **input langsung** untuk algoritma clustering (KMeans, Hierarchical, DBSCAN, dll.).

In [ ]:
# ===========================================================================
# Buat DataFrame Final: teams_ml_features
# ===========================================================================

teams_ml_features = df_scores.copy()

# Urutkan berdasarkan nama klub (konsistensi)
teams_ml_features = teams_ml_features.sort_values("club").reset_index(drop=True)

print("📦 Dataset Final: teams_ml_features")
print("=" * 50)
print(f"   Shape : {teams_ml_features.shape}")
print(f"   Kolom : {list(teams_ml_features.columns)}")
print(f"   Tipe  : {teams_ml_features.dtypes.value_counts().to_dict()}")
print()
teams_ml_features

In [ ]:
# --- Statistik Deskriptif Dataset Final ---
teams_ml_features.describe().round(2)

In [ ]:
# --- Korelasi antar Style Scores ---
corr = teams_ml_features[score_cols].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="📊 Correlation Matrix — Style Scores",
    aspect="auto",
)

fig.update_layout(width=750, height=650, font=dict(size=10))
fig.show()

### 💾 Simpan Dataset ke CSV

Dataset final disimpan ke file CSV agar bisa diakses oleh notebook clustering tanpa perlu mengulang seluruh proses feature engineering.

In [ ]:
# --- Simpan ke CSV ---
output_dir = PROJECT_ROOT / "data" / "ml_ready"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "teams_ml_features.csv"
teams_ml_features.to_csv(output_path, index=False)

print(f"✅ Dataset disimpan ke: {output_path}")
print(f"   Size: {output_path.stat().st_size / 1024:.1f} KB")

---

## ✅ Kesimpulan

Notebook ini telah menyelesaikan 3 tahap utama:

### 1. Feature Selection
- Memilih **fitur-fitur relevan** dari 124+ kolom berdasarkan konsep taktis sepakbola.
- Fitur dikelompokkan menjadi **9 dimensi gaya bermain**.

### 2. Feature Engineering
- Menormalisasi semua fitur menggunakan **Min-Max Scaling** ke skala 0–100.
- Menangani fitur *inverted* (dimana nilai tinggi = performa buruk).

### 3. Style Scoring
- Menghitung **9 skor komposit** menggunakan simple average.
- Memvalidasi hasil melalui ranking dan visualisasi radar chart.

### Output
- File: `data/ml_ready/teams_ml_features.csv`
- Berisi: 20 baris (klub) × 10 kolom (club + 9 style scores)
- Siap digunakan pada **notebook clustering berikutnya**.

---

*Notebook berikutnya: Clustering Analysis — Mengelompokkan tim berdasarkan style scores untuk mengidentifikasi pola gaya bermain di LaLiga.*